# Continuation Ledger — build 2

**Dean Kulik — QuHarmonics Research Group** · CC BY-NC 4.0

Input: a system (states, actions, transition map) and an insufficient reader $R$. Output: the minimum state the reader must add so that every continuation is determined:

$$R^\star(x_1)=R^\star(x_2)\;\Longrightarrow\;R^\star(T_a x_1)=R^\star(T_a x_2)\quad\text{for every action }a.$$

**What build 2 adds**

- **BUILD.** `close_reader` accepts a branching source: `T(x, a)` may return `Choices({...})`. Two states then share a class only when, for every action, they reach the same set of classes (bisimulation refinement). On a deterministic source this is exactly build 1's Moore refinement.
- **FIX.** A witness now observes refusals. After a word, the observation is the set of reader values the source may show, plus `refused` if some run may refuse the last move. The first sweep turned up 8 split fibers with no witness word under the old observation. This also removes a build-1 edge case, where two states that both refused a move were counted as told apart.
- **TEST 5.** Is "capacity = ledger" the general receiver law?
- Tests 1–4 are rerun unchanged against the new module as regression.

In [1]:
# ============================================================ continuation_ledger (the module)
import itertools, math, random
from collections import defaultdict, deque

class Ledger:
    """Result of closing a reader. All fields are computed, none are supplied."""
    def __repr__(self):
        return (f"Ledger(states={len(self.states)}, reader_fibers={len(self.reader_fibers)}, "
                f"refined_classes={self.n_classes}, required_capacity={self.required_capacity}, "
                f"ledger_bits={self.ledger_bits}, violations={self.violations})")

class Choices(frozenset):
    """T(x, a) returns Choices({...}) when the source may continue in more than one way."""

def close_reader(states, actions, T, R):
    """Coarsest refinement R* of reader R that is closed under every action.
    T(x, a) returns the successor state, None if a is not legal at x, or Choices({...}) when the
    source is nondeterministic. For deterministic T this is Moore refinement; for Choices it is
    bisimulation refinement: equal R* means equal reader value and, per action, the same set of
    successor classes."""
    L = Ledger(); L.states = list(states); L.actions = list(actions)
    idx = {x: i for i, x in enumerate(L.states)}
    succ = []
    for x in L.states:
        row = []
        for a in L.actions:
            y = T(x, a)
            row.append(() if y is None else tuple(sorted(idx[z] for z in y)) if isinstance(y, Choices) else (idx[y],))
        succ.append(row)
    rv = [R(x) for x in L.states]
    first = {}
    blk = [first.setdefault(v, len(first)) for v in rv]
    while True:                                   # refine until stable
        sig = {}
        new = [sig.setdefault((blk[i], tuple(frozenset(blk[j] for j in s) for s in succ[i])), len(sig))
               for i in range(len(L.states))]
        if len(sig) == len(set(blk)):
            blk = new; break
        blk = new
    L.block, L.reader_value, L._succ, L._idx = blk, rv, succ, idx
    L.n_classes = len(set(blk))
    # reader fibers and the classes hidden inside each
    fib = defaultdict(list)
    for i, v in enumerate(rv): fib[v].append(i)
    L.reader_fibers = dict(fib)
    L.future_classes = {v: sorted({blk[i] for i in ids}) for v, ids in fib.items()}
    L.required_capacity = max(len(c) for c in L.future_classes.values())
    L.ledger_bits = math.ceil(math.log2(L.required_capacity)) if L.required_capacity > 1 else 0
    # minimum labels: the index of a state's class inside its own fiber
    L.minimum_labels = {L.states[i]: L.future_classes[rv[i]].index(blk[i]) for i in range(len(L.states))}
    L.refined_reader = lambda x: (R(x), L.minimum_labels[x])
    # reduced transitions on classes, and the closure check
    red = defaultdict(set)
    for i in range(len(L.states)):
        for k, s in enumerate(succ[i]):
            red[(blk[i], k)].add(frozenset(blk[j] for j in s))
    L.violations = sum(1 for v in red.values() if len(v) > 1)
    one = lambda f: None if not f else next(iter(f)) if len(f) == 1 else f
    L.reduced_transitions = {key: one(next(iter(v))) for key, v in red.items()}
    L.class_value = {blk[i]: rv[i] for i in range(len(L.states))}
    # one shortest witness per split fiber: same reader value, different lawful future
    L.branching_counterexamples = []
    for v, ids in fib.items():
        cl = L.future_classes[v]
        if len(cl) < 2: continue
        i1 = next(i for i in ids if blk[i] == cl[0]); i2 = next(i for i in ids if blk[i] == cl[1])
        L.branching_counterexamples.append(_witness(L, i1, i2))
    return L

def _witness(L, i1, i2):
    """shortest action word after which the ORIGINAL reader tells i1 and i2 apart. With a branching
    source the observation after a word is the set of reader values the source may show, plus
    'refused' if some run may refuse the last action."""
    start = (frozenset([i1]), frozenset([i2])); seen = {start}; q = deque([(start, ())])
    while q:
        (A, B), w = q.popleft()
        for k in range(len(L.actions)):
            nA = frozenset(j for i in A for j in L._succ[i][k]); nB = frozenset(j for i in B for j in L._succ[i][k])
            oA = {L.reader_value[j] for j in nA} | ({"refused"} if any(not L._succ[i][k] for i in A) else set())
            oB = {L.reader_value[j] for j in nB} | ({"refused"} if any(not L._succ[i][k] for i in B) else set())
            if oA != oB: return (L.states[i1], L.states[i2], w + (L.actions[k],))
            if nA and (nA, nB) not in seen:
                seen.add((nA, nB)); q.append(((nA, nB), w + (L.actions[k],)))
    return (L.states[i1], L.states[i2], None)

def peek(L, cls):
    """what the original reader sees in a refined class"""
    return L.class_value[cls]

def poke(L, cls, word):
    """apply an action word to a refined class through the reduced table; no history replay.
    Returns the class, a frozenset of classes if the source branched, or None if illegal."""
    k = {a: n for n, a in enumerate(L.actions)}; cur = {cls}
    for a in word:
        nxt = set()
        for c in cur:
            r = L.reduced_transitions[(c, k[a])]
            if r is not None: nxt |= r if isinstance(r, frozenset) else {r}
        if not nxt: return None
        cur = nxt
    return next(iter(cur)) if len(cur) == 1 else frozenset(cur)

def label(L, x):
    return L.block[L._idx[x]]

def future_classes(readings, completions, out):
    """Myhill-Nerode classes of readings: equal iff every completion gives the same output on the
    positions still pending. Positions already determined count as emitted, not carried."""
    classes = set()
    for r in readings:
        rows = [out(c) for c in completions(r)]
        pend = tuple(i for i in range(len(rows[0])) if len({o[i] for o in rows}) > 1)
        classes.add((pend, tuple(tuple(o[i] for i in pend) for o in rows)))
    return classes

def transport(sources, image, targets):
    """write a source into a receiver: image(x) -> target or None (does not fit).
    Returns excluded sources, invented targets (never written) and identified targets."""
    pre = defaultdict(list); excluded = 0
    for x in sources:
        t = image(x)
        if t is None: excluded += 1
        else: pre[t].append(x)
    invented = sum(1 for t in targets if t not in pre)
    identified = sum(1 for t in targets if len(pre.get(t, ())) > 1)
    verdict = "exclusion" if excluded else ("invention" if invented else "faithful")
    return dict(excluded=excluded, invented=invented, identified=identified, verdict=verdict)

RESULTS = []
def contract(name, ok, detail):
    RESULTS.append((name, ok, detail)); print(f"{'PASS' if ok else 'FAIL'}  {name}: {detail}")
print("continuation_ledger loaded (build 2)")

continuation_ledger loaded (build 2)


---
## Test 1 — a deliberately lossy machine

A 2-bit counter plus one junk bit that never affects anything. Actions: `inc` (counter + 1 mod 4) and `flip` (toggle the junk bit). The reader sees only the counter's high bit.

**Contract:** the runtime splits each fiber on the counter's low bit, which is a 1-bit ledger. It must **not** add the junk bit, because that distinction has no effect on any future. `poke` through the reduced table must agree with running the machine.

In [2]:
states = [(c, j) for c in range(4) for j in (0, 1)]
T1 = lambda x, a: ((x[0] + 1) % 4, x[1]) if a == "inc" else (x[0], x[1] ^ 1)
L1 = close_reader(states, ["inc", "flip"], T1, R=lambda x: x[0] // 2)
print(L1)
print("witnesses (same reading, different future):", L1.branching_counterexamples)
low_bit = all(L1.minimum_labels[x] == L1.minimum_labels[(x[0] % 2, 0) if x[0] < 2 else (2 + x[0] % 2, 0)] and
              len({L1.minimum_labels[y] for y in states if y[0] // 2 == x[0] // 2 and y[0] % 2 == x[0] % 2}) == 1
              for x in states) and len({(x[0] // 2, L1.minimum_labels[x]) for x in states}) == 4
junk_merged = all(label(L1, (c, 0)) == label(L1, (c, 1)) for c in range(4))
rng = random.Random(1); ok = 0
for _ in range(2000):
    x = rng.choice(states); w = tuple(rng.choice(["inc", "flip"]) for _ in range(rng.randint(0, 12))); y = x
    for a in w: y = T1(y, a)
    ok += poke(L1, label(L1, x), w) == label(L1, y)
contract("T1 ledger = 1 bit", L1.ledger_bits == 1 and L1.violations == 0, f"ledger_bits={L1.ledger_bits}, violations={L1.violations}")
contract("T1 ledger is the low counter bit", low_bit, f"classes = (high bit, low bit): {low_bit}")
contract("T1 junk bit not added", junk_merged, f"states differing only in the junk bit share a class: {junk_merged}")
contract("T1 poke = run", ok == 2000, f"{ok}/2000 random words")

Ledger(states=8, reader_fibers=2, refined_classes=4, required_capacity=2, ledger_bits=1, violations=0)
witnesses (same reading, different future): [((0, 0), (1, 0), ('inc',)), ((2, 0), (3, 0), ('inc',))]
PASS  T1 ledger = 1 bit: ledger_bits=1, violations=0
PASS  T1 ledger is the low counter bit: classes = (high bit, low bit): True
PASS  T1 junk bit not added: states differing only in the junk bit share a class: True
PASS  T1 poke = run: 2000/2000 random words


---
## Test 2 — binary adder

The states are every input history of bit pairs $(a_i, b_i)$ read least-significant-first, up to length 6. There are 5461 of them, and the machine is never told the word "carry". The reader sees only the sum bits written so far.

**Contract:** the runtime finds a 1-bit ledger. On every history that still has a future, its classes coincide exactly with (written bits, carry). On histories with no future (length 6) it adds nothing.

In [3]:
LEN = 6
def written(h):
    c, out = 0, []
    for a, b in h:
        v = a + b + c; out.append(v % 2); c = v // 2
    return tuple(out), c
hist = [h for k in range(LEN + 1) for h in itertools.product([(0, 0), (0, 1), (1, 0), (1, 1)], repeat=k)]
T2 = lambda h, a: h + (a,) if len(h) < LEN else None
L2 = close_reader(hist, [(0, 0), (0, 1), (1, 0), (1, 1)], T2, R=lambda h: written(h)[0])
print(L2)
print("a witness:", L2.branching_counterexamples[-1])
live = [h for h in hist if len(h) < LEN]; dead = [h for h in hist if len(h) == LEN]
same = (len({label(L2, h) for h in live}) == len({(written(h)[0], written(h)[1]) for h in live}) ==
        len({(label(L2, h), written(h)[1]) for h in live}))
nothing_at_end = all(len(L2.future_classes[written(h)[0]]) == 1 for h in dead)
rng = random.Random(2); ok = 0
for _ in range(2000):
    h = rng.choice(live); w = tuple(rng.choice([(0, 0), (0, 1), (1, 0), (1, 1)]) for _ in range(rng.randint(0, LEN - len(h))))
    ok += poke(L2, label(L2, h), w) == label(L2, h + w) and peek(L2, label(L2, h + w)) == written(h + w)[0]
contract("T2 ledger = 1 bit", L2.ledger_bits == 1 and L2.violations == 0, f"ledger_bits={L2.ledger_bits}, violations={L2.violations}")
contract("T2 ledger is the carry", same, f"classes on live histories == (written bits, carry): {same}")
contract("T2 no ledger without a future", nothing_at_end, f"length-{LEN} fibers unsplit: {nothing_at_end}")
contract("T2 peek/poke = run", ok == 2000, f"{ok}/2000 random continuations")

Ledger(states=5461, reader_fibers=127, refined_classes=184, required_capacity=2, ledger_bits=1, violations=0)
a witness: (((0, 1), (0, 1), (0, 1), (0, 1), (0, 0)), ((0, 1), (0, 1), (0, 1), (0, 1), (1, 1)), ((0, 0),))
PASS  T2 ledger = 1 bit: ledger_bits=1, violations=0
PASS  T2 ledger is the carry: classes on live histories == (written bits, carry): True
PASS  T2 no ledger without a future: length-6 fibers unsplit: True
PASS  T2 peek/poke = run: 2000/2000 random continuations


---
## Test 3 — reader-cut adder: when do gap ledgers factor?

This is the corpus machine from "Carry Capacity and the Conditional Cut Law". It has radix $b$ and $m$ operands, and $t_q$ is the digit sum at position $q$. Each step computes $s_q = (t_q + c_q) \bmod b$ and $c_{q+1} = \lfloor (t_q + c_q)/b \rfloor$. A reader reads the positions in a cut $S$, and its state count is $F(S)$, computed by `future_classes`. The ledgers are *gap-local* when $F(S)$ equals the product over the maximal read runs.

**Contract:** for each $(b, m)$, the smallest gap at which factoring holds for every cut equals $\lceil \log_b m \rceil$. The failure counts must reproduce the corpus exactly: 18 for $(2,3)$, 27 for $(2,4)$, 2 for $(3,4)$ and 2 for $(4,5)$.

In [4]:
def line_out(t, b):
    c, s = 0, []
    for v in t:
        v += c; s.append(v % b); c = v // b
    return tuple(s)
def runs_of(S, n):
    out, cur = [], None
    for p in range(n):
        if p in S:
            cur = [p, p] if cur is None else [cur[0], p]
        elif cur is not None:
            out.append(tuple(cur)); cur = None
    if cur is not None: out.append(tuple(cur))
    return out
def gaps_of(S, n):
    U = [p for p in range(n) if p not in S]
    return [len(list(g)) for _, g in itertools.groupby(enumerate(U), lambda t: t[1] - t[0])]
def make_F(n, b, m):
    hi = m * (b - 1) + 1; memo = {}
    def F(S):
        S = tuple(sorted(S))
        if S not in memo:
            U = [p for p in range(n) if p not in S]
            def completions(r):
                for uv in itertools.product(range(hi), repeat=len(U)):
                    t = [0] * n
                    for p, v in zip(S, r): t[p] = v
                    for p, v in zip(U, uv): t[p] = v
                    yield t
            memo[S] = len(future_classes(itertools.product(range(hi), repeat=len(S)), completions, lambda t: line_out(t, b)))
        return memo[S]
    return F
CORPUS_FAILS = {(2, 2): 0, (2, 3): 18, (2, 4): 27, (3, 3): 0, (3, 4): 2, (4, 5): 2, (5, 5): 0}
print(f"{'b':>2} {'m':>2} {'n':>2} {'ceil(log_b m)':>13} {'observed':>9} {'fails':>6} {'corpus':>7}")
all_ok = True
for (b, m, n) in [(2, 2, 6), (2, 3, 6), (2, 4, 6), (3, 3, 5), (3, 4, 4), (4, 5, 4), (5, 5, 3)]:
    F = make_F(n, b, m); by = defaultdict(lambda: [0, 0])
    for k in range(n + 1):
        for S in itertools.combinations(range(n), k):
            prod = 1
            for (l, r) in runs_of(set(S), n): prod *= F(range(l, r + 1))
            gs = gaps_of(S, n); g = min(gs) if gs else n
            by[g][0 if prod == F(S) else 1] += 1
    obs = min(g for g in sorted(by) if all(by[gg][1] == 0 for gg in by if gg >= g))
    pred = math.ceil(math.log(m, b) - 1e-12); fails = sum(v[1] for v in by.values())
    ok = obs == pred and fails == CORPUS_FAILS[(b, m)]; all_ok &= ok
    print(f"{b:>2} {m:>2} {n:>2} {pred:>13} {obs:>9} {fails:>6} {CORPUS_FAILS[(b, m)]:>7}  {'ok' if ok else 'MISS'}")
contract("T3 factoring threshold", all_ok, "observed gap threshold == ceil(log_b m) and failure counts == corpus, 7 radix-operand pairs")

 b  m  n ceil(log_b m)  observed  fails  corpus
 2  2  6             1         1      0       0  ok
 2  3  6             2         2     18      18  ok
 2  4  6             2         2     27      27  ok
 3  3  5             1         1      0       0  ok
 3  4  4             2         2      2       2  ok
 4  5  4             2         2      2       2  ok
 5  5  3             1         1      0       0  ok
PASS  T3 factoring threshold: observed gap threshold == ceil(log_b m) and failure counts == corpus, 7 radix-operand pairs


---
## Test 4 — arena contraction: receiver capacity against the written ledger

The arena is $K = 3$ tokens on a 6-cycle, with exclusion (at most one token per vertex). Contract the edge $(0,1)$ into one site $w$ of capacity $c$. The runtime computes $\ell$, the most the source ever writes into $w$; it is not told $\ell$. It then writes every source configuration into the receiver.

**Contract:** $c<\ell$ gives exclusion (sources with no image). $c=\ell$ gives faithful transport: nothing excluded, nothing invented, and merged states counted as identification. $c>\ell$ gives invention (receiver states no source writes).

In [5]:
n, K, u, v = 6, 3, 0, 1
src = list(itertools.combinations(range(n), K))
rest_sites = [x for x in range(n) if x not in (u, v)]
load = lambda S: sum(1 for x in S if x in (u, v))
ell = max(load(S) for S in src)
print(f"runtime-computed ledger into the merged site: ell = {ell}")
verdicts = {}
for c in (1, 2, 3):
    targets = [(r, k) for k in range(c + 1) for r in itertools.combinations(rest_sites, K - k)]
    img = lambda S, c=c: (tuple(x for x in S if x not in (u, v)), load(S)) if load(S) <= c else None
    res = transport(src, img, targets); verdicts[c] = res["verdict"]
    print(f"  capacity c={c}: {res}")
expect = {c: ("exclusion" if c < ell else "faithful" if c == ell else "invention") for c in (1, 2, 3)}
contract("T4 capacity rule", verdicts == expect, f"verdicts {verdicts} == expected {expect}")

runtime-computed ledger into the merged site: ell = 2
  capacity c=1: {'excluded': 4, 'invented': 0, 'identified': 6, 'verdict': 'exclusion'}
  capacity c=2: {'excluded': 0, 'invented': 0, 'identified': 6, 'verdict': 'faithful'}
  capacity c=3: {'excluded': 0, 'invented': 1, 'identified': 6, 'verdict': 'invention'}
PASS  T4 capacity rule: verdicts {1: 'exclusion', 2: 'faithful', 3: 'invention'} == expected {1: 'exclusion', 2: 'faithful', 3: 'invention'}


---
## Test 5 — is "capacity = ledger" the general receiver law?

**Arena.** $K$ tokens sit on a connected graph $X$, at most one per vertex. A connected vertex set $M$ is contracted into one receiver site $w$ of capacity $c$. $M$ is either one edge or an edge orbit such as a 3-vertex path. The **reader** is the receiver's own configuration: the tokens outside $M$ and the number of tokens at $w$. The **actions** are the receiver's moves. The source makes a receiver move by any source move that maps to it, so the source can branch. The runtime computes $\ell$, the most tokens the source ever writes into $M$ (the collision multiplicity); it is not told $\ell$.

At each capacity $c$ the runtime counts five defects:

| count | meaning |
|---|---|
| excluded | source configurations with no receiver image |
| unmapped | source moves whose image is not the matching receiver move (moves inside $M$ map to identity) |
| split | receiver readings whose source preimages have different futures (`close_reader` split fibers) |
| target-only | receiver moves from a written state that no preimage can make |
| invented | receiver states the source never writes |

The contraction is **faithful** at $c$ when all five are zero. $\ell_{\min}$ is the smallest faithful $c$.

**Declared before any run**

- **T5a hosting** (the build-1 law, now across the family): $c<\ell$ excludes. At $c=\ell$ nothing is excluded, invented or unmapped. $c>\ell$ invents, with a target-only move.
- **T5b the review's prediction:** $\ell_{\min}=\ell$ in every case. That is, occupancy at capacity $\ell$ is a continuation-faithful ledger.
- **T5c the declared alternative:** $\ell_{\min}=\ell$ exactly when every vertex outside $M$ is adjacent to all of $M$ or to none of it. In graph theory this makes $M$ a *module*; for one edge it means the endpoints are twins. Otherwise `close_reader` splits a fiber, a single move exposes the split, and no capacity is faithful.

**Set before the 600-case sweep**

- **T5d ledger shape:** group the vertices of $M$ by what they see outside $M$. The ledger `close_reader` recovers is the token count in each group: the reader (outside tokens, count per group) is closed and has exactly the classes of $R^\star$.
- **T5e secondary:** for $c>\ell$ a target-only class is detected, and the shortest witness is one receiver move from a written state.

In [6]:
# ============================================================ arena adapter (test 5)
import collections
def arena(n, E, M, K):
    """K exclusion tokens on the graph (n, E). The connected vertex set M is contracted to one site w.
    Reader: the receiver's configuration (tokens outside M, tokens at w). Actions: the receiver's moves.
    The source realizes a receiver move by ANY source move that maps to it, so it may branch."""
    E = {frozenset(e) for e in E}; M = frozenset(M)
    adj = {x: {y for e in E if x in e for y in e if y != x} for x in range(n)}
    outs = [x for x in range(n) if x not in M]
    ext = {m: frozenset(adj[m] - M) for m in M}                      # what each M-vertex sees outside M
    classes = [sorted(m for m in M if ext[m] == s) for s in sorted(set(ext.values()), key=lambda s: min(m for m in M if ext[m] == s))]
    tadj = {x: {y for y in adj[x] if y not in M} | ({"w"} if adj[x] & M else set()) for x in outs}
    tadj["w"] = {y for y in outs if adj[y] & M}
    tacts = [(p, q) for p in sorted(tadj, key=str) for q in sorted(tadj[p], key=str)]
    src = [frozenset(S) for S in itertools.combinations(range(n), K)]
    pi = lambda S: (frozenset(S - M), len(S & M))
    def T(S, a):
        p, q = a
        if p != "w" and q != "w":
            return S - {p} | {q} if p in S and q not in S else None
        if q == "w":
            mv = [S - {p} | {z} for z in M if z in adj[p] and z not in S] if p in S else []
        else:
            mv = [S - {m} | {q} for m in M if q in adj[m] and m in S] if q not in S else []
        return Choices(mv) if mv else None
    return dict(n=n, adj=adj, M=M, K=K, outs=outs, classes=classes, module=len(classes) == 1,
                tacts=tacts, src=src, pi=pi, T=T, ell=max(len(S & M) for S in src))

def receiver(A, c):
    """w is one site that holds up to c tokens"""
    K, outs = A["K"], A["outs"]
    states = [(frozenset(r), K - len(r)) for k in range(c + 1) if 0 <= K - k <= len(outs)
              for r in itertools.combinations(outs, K - k)]
    def Tt(t, a):
        (r, k), (p, q) = t, a
        if p != "w" and q != "w": return (r - {p} | {q}, k) if p in r and q not in r else None
        if q == "w": return (r - {p}, k + 1) if p in r and k < c else None
        return (r | {q}, k - 1) if k > 0 and q not in r else None
    return states, Tt

def assess(A, c, L):
    """source against the receiver at capacity c; faithful = every count below is zero"""
    src, pi, T, adj, M = A["src"], A["pi"], A["T"], A["adj"], A["M"]
    tstates, Tt = receiver(A, c)
    img = lambda S: pi(S) if len(S & M) <= c else None
    tr = transport(src, img, tstates)
    unmapped = 0                         # source moves with no matching receiver move (moves inside M map to identity)
    for S in src:
        if img(S) is None: continue
        for p in S:
            for q in adj[p]:
                if q in S or (p in M and q in M): continue
                S2 = S - {p} | {q}; a = ("w" if p in M else p, "w" if q in M else q)
                if img(S2) is None or Tt(img(S), a) != img(S2): unmapped += 1
    pre = defaultdict(list)
    for S in src:
        if img(S) is not None: pre[img(S)].append(S)
    target_only = 0                      # receiver moves from a written state that no preimage can make
    for t, Ss in pre.items():
        for a in A["tacts"]:
            if Tt(t, a) is not None and all(T(S, a) is None for S in Ss): target_only += 1
    split = len(L.branching_counterexamples)
    faithful = not (tr["excluded"] or tr["invented"] or unmapped or split or target_only)
    return dict(c=c, excluded=tr["excluded"], invented=tr["invented"], unmapped=unmapped, split=split,
                target_only=target_only, faithful=faithful, written=set(pre), Tt=Tt)

def target_only_witness(A, R):
    """shortest receiver word from a written state to a state the source never writes"""
    q = deque((t, t, ()) for t in sorted(R["written"], key=str)); seen = set(R["written"])
    while q:
        t0, t, w = q.popleft()
        for a in A["tacts"]:
            t2 = R["Tt"](t, a)
            if t2 is None or t2 in seen: continue
            if t2 not in R["written"]: return t0, w + (a,), t2
            seen.add(t2); q.append((t0, t2, w + (a,)))

def run_case(n, E, M, K):
    A = arena(n, E, M, K); L = close_reader(A["src"], A["tacts"], A["T"], A["pi"])
    ell = A["ell"]; rows = {c: assess(A, c, L) for c in range(1, K + 1)}
    Rcls = lambda S: (frozenset(S - A["M"]), tuple(len(S & set(c)) for c in A["classes"]))
    Lcls = close_reader(A["src"], A["tacts"], A["T"], Rcls)
    shape = (Lcls.required_capacity == 1 and
             len({(label(L, S), Rcls(S)) for S in A["src"]}) == len({Rcls(S) for S in A["src"]}) == L.n_classes)
    return dict(A=A, L=L, ell=ell, rows=rows, module=A["module"],
                lmin=next((c for c in rows if rows[c]["faithful"]), None),
                wlen=[len(w[2]) if w[2] is not None else None for w in L.branching_counterexamples],
                shape=shape, witness=target_only_witness(A, rows[ell + 1]) if ell < K else None)

def hosting_ok(r):
    ell, rows = r["ell"], r["rows"]
    return all(rows[c]["excluded"] > 0 if c < ell else
               rows[c]["excluded"] == rows[c]["invented"] == rows[c]["unmapped"] == 0 if c == ell else
               rows[c]["invented"] > 0 and rows[c]["target_only"] > 0 for c in rows)

def connected(n, E):
    adj = {x: set() for x in range(n)}
    for a, b in E: adj[a].add(b); adj[b].add(a)
    seen, st = {0}, [0]
    while st:
        for y in adj[st.pop()] - seen: seen.add(y); st.append(y)
    return len(seen) == n, adj

def random_case(rng, force_module):
    while True:
        n = rng.choice([5, 6, 7]); p = rng.choice([0.35, 0.5, 0.65])
        E = [e for e in itertools.combinations(range(n), 2) if rng.random() < p]
        ok, adj = connected(n, E)
        if not ok: continue
        M = {rng.randrange(n)}; size = rng.choice([2, 3])
        while len(M) < size:
            M.add(rng.choice(sorted({y for x in M for y in adj[x]} - M)))
        if force_module:                 # rewire so every vertex outside M sees all of M or none of it
            N = {x for x in range(n) if x not in M and rng.random() < 0.5} or {rng.choice([x for x in range(n) if x not in M])}
            E = [e for e in E if not ((e[0] in M) ^ (e[1] in M))] + [tuple(sorted((m, x))) for m in M for x in N]
            ok, _ = connected(n, E)
            if not ok: continue
        return n, E, M, rng.choice([k for k in (2, 3, 4) if k <= n - 1])

In [7]:
cyc = lambda n: [(i, (i + 1) % n) for i in range(n)]
PET = [(i, (i + 1) % 5) for i in range(5)] + [(i, i + 5) for i in range(5)] + [(5 + i, 5 + (i + 2) % 5) for i in range(5)]
NAMED = {"path P5, e=(1,2)": (5, [(0,1),(1,2),(2,3),(3,4)], {1,2}),
         "cycle C6, e=(0,1)": (6, cyc(6), {0,1}),
         "K4, e=(0,1)": (4, list(itertools.combinations(range(4), 2)), {0,1}),
         "diamond, e=(0,1)": (4, [(0,1),(0,2),(0,3),(1,2),(1,3)], {0,1}),
         "diamond, e=(0,2)": (4, [(0,1),(0,2),(0,3),(1,2),(1,3)], {0,2}),
         "star K1,4, e=(0,1)": (5, [(0,1),(0,2),(0,3),(0,4)], {0,1}),
         "twin+tail, e=(0,1)": (5, [(0,1),(0,2),(0,3),(1,2),(1,3),(2,4)], {0,1}),
         "Petersen, e=(0,1)": (10, PET, {0,1}),
         "K5, M=path 0-1-2": (5, list(itertools.combinations(range(5), 2)), {0,1,2}),
         "C6, M=path 0-1-2": (6, cyc(6), {0,1,2}),
         "P3 module, M=0-1-2": (6, [(0,1),(1,2)] + [(m,x) for m in (0,1,2) for x in (3,4)] + [(3,4),(4,5)], {0,1,2})}
def verdict(x):
    f = [s for s, k in (("excl", "excluded"), ("split", "split"), ("invent", "invented")) if x[k]]
    return "+".join(f) or "FAITHFUL"
named = []
print(f"{'case':<20}{'K':>2}{'ell':>4}  {'groups of M':<17}{'receiver at c = 1 .. K':<42}{'ell_min':>8}  ledger")
for name, (n, E, M) in NAMED.items():
    for K in (2, 3):
        if K >= n: continue
        r = run_case(n, E, M, K); r["name"] = name; named.append(r); L = r["L"]
        v = "  ".join(f"{c}:{verdict(x)}" for c, x in r["rows"].items())
        led = "occupancy" if L.required_capacity == 1 else f"occupancy + {L.ledger_bits} bit ({L.required_capacity} classes)"
        print(f"{name:<20}{K:>2}{r['ell']:>4}  {str(r['A']['classes']):<17}{v:<42}{str(r['lmin']):>8}  {led}")

case                 K ell  groups of M      receiver at c = 1 .. K                     ell_min  ledger
path P5, e=(1,2)     2   2  [[1], [2]]       1:excl+split  2:split                         None  occupancy + 1 bit (2 classes)
path P5, e=(1,2)     3   2  [[1], [2]]       1:excl+split  2:split  3:split+invent         None  occupancy + 1 bit (2 classes)
cycle C6, e=(0,1)    2   2  [[0], [1]]       1:excl+split  2:split                         None  occupancy + 1 bit (2 classes)
cycle C6, e=(0,1)    3   2  [[0], [1]]       1:excl+split  2:split  3:split+invent         None  occupancy + 1 bit (2 classes)
K4, e=(0,1)          2   2  [[0, 1]]         1:excl  2:FAITHFUL                               2  occupancy
K4, e=(0,1)          3   2  [[0, 1]]         1:excl  2:FAITHFUL  3:invent                     2  occupancy
diamond, e=(0,1)     2   2  [[0, 1]]         1:excl  2:FAITHFUL                               2  occupancy
diamond, e=(0,1)     3   2  [[0, 1]]         1:excl  2:FAITHFUL  3:

A split and its witness, as the runtime emits them. Then a target-only class for $c>\ell$ and its shortest witness.

In [8]:
def show(S): return "{" + ",".join(map(str, sorted(S))) + "}"
for r in named:
    if r["name"] in ("path P5, e=(1,2)", "diamond, e=(0,2)", "C6, M=path 0-1-2") and r["A"]["K"] == 2:
        A, (S1, S2, w) = r["A"], r["L"].branching_counterexamples[0]
        a = w[0]; say = lambda S: "legal" if A["T"](S, a) is not None else "refused"
        rest, load = A["pi"](S1)
        print(f"{r['name']}: {show(S1)} and {show(S2)} read the same (outside {show(rest)}, w holds {load}).")
        print(f"    move {a[0]}->{a[1]}: {say(S1)} from {show(S1)}, {say(S2)} from {show(S2)}; the receiver at c={r['ell']} says legal.")
r = next(r for r in named if r["name"] == "path P5, e=(1,2)" and r["A"]["K"] == 3)
(t0, w, t1) = r["witness"]
print(f"\npath P5, K=3, c=3: written state (outside {show(t0[0])}, w holds {t0[1]})  --{w[0][0]}->{w[0][1]}-->  "
      f"(outside {show(t1[0])}, w holds {t1[1]}), which no source configuration writes")

path P5, e=(1,2): {0,1} and {0,2} read the same (outside {0}, w holds 1).
    move 0->w: refused from {0,1}, legal from {0,2}; the receiver at c=2 says legal.
diamond, e=(0,2): {0,1} and {1,2} read the same (outside {1}, w holds 1).
    move w->3: legal from {0,1}, refused from {1,2}; the receiver at c=2 says legal.
C6, M=path 0-1-2: {0,1} and {0,2} read the same (outside {}, w holds 2).
    move w->3: refused from {0,1}, legal from {0,2}; the receiver at c=2 says legal.

path P5, K=3, c=3: written state (outside {0}, w holds 2)  --0->w-->  (outside {}, w holds 3), which no source configuration writes


### The sweep

600 random cases, seeded: connected graphs on 5–7 vertices, $M$ a random connected 2- or 3-vertex set, $K\in\{2,3,4\}$. The first 300 are unconstrained. In the last 300 the edges between $M$ and the rest are rewired so that $M$ is a module, so both sides of T5c are tested.

In [9]:
rng = random.Random(5)
sweep = [run_case(*random_case(rng, fm)) for fm in [False] * 300 + [True] * 300]
allc = named + sweep
mods = [r for r in allc if r["module"]]; non = [r for r in allc if not r["module"]]
over = [r for r in allc if r["ell"] < r["A"]["K"]]
print(f"cases {len(allc)}: {len(mods)} with M a module, {len(non)} without")
print("cases by (ell, groups of M):", dict(sorted(collections.Counter((r["ell"], len(r["A"]["classes"])) for r in allc).items())))
print(f"module cases faithful at c=ell: {sum(r['lmin'] == r['ell'] for r in mods)}/{len(mods)}")
print(f"non-module cases faithful at any c: {sum(r['lmin'] is not None for r in non)}/{len(non)}; "
      f"shortest split witness lengths: {dict(collections.Counter(min(r['wlen']) for r in non))}")
contract("T5a hosting law", all(map(hosting_ok, allc)),
         f"{sum(map(hosting_ok, allc))}/{len(allc)}: c<ell excludes, c=ell hosts with 0 excluded/invented/unmapped, c>ell invents")
contract("T5b ell_min = ell (review)", all(r["lmin"] == r["ell"] for r in allc),
         f"{sum(r['lmin'] == r['ell'] for r in allc)}/{len(allc)}; on {len(non)} cases occupancy at c=ell still splits")
contract("T5c ell_min = ell iff module", all((r["lmin"] == r["ell"]) == r["module"] for r in allc)
         and all(not r["wlen"] for r in mods) and all(min(r["wlen"]) == 1 for r in non),
         f"{sum((r['lmin'] == r['ell']) == r['module'] for r in allc)}/{len(allc)}; every non-module split exposed by one move")
contract("T5d ledger = count per group", all(r["shape"] for r in allc),
         f"{sum(r['shape'] for r in allc)}/{len(allc)}: (outside, count per group of M) is closed and equals R*")
contract("T5e c>ell target-only class", all(r["witness"] and len(r["witness"][1]) == 1 and r["rows"][r["ell"] + 1]["target_only"] > 0 for r in over),
         f"{len(over)} cases with c>ell possible; all detected, shortest witness 1 move")

cases 622: 322 with M a module, 300 without
cases by (ell, groups of M): {(2, 1): 229, (2, 2): 170, (2, 3): 27, (3, 1): 93, (3, 2): 45, (3, 3): 58}
module cases faithful at c=ell: 322/322
non-module cases faithful at any c: 0/300; shortest split witness lengths: {1: 300}
PASS  T5a hosting law: 622/622: c<ell excludes, c=ell hosts with 0 excluded/invented/unmapped, c>ell invents
FAIL  T5b ell_min = ell (review): 322/622; on 300 cases occupancy at c=ell still splits
PASS  T5c ell_min = ell iff module: 622/622; every non-module split exposed by one move
PASS  T5d ledger = count per group: 622/622: (outside, count per group of M) is closed and equals R*
PASS  T5e c>ell target-only class: 304 cases with c>ell possible; all detected, shortest witness 1 move


---
## Result

In [10]:
print(f"{'contract':<40} result")
print("-" * 52)
for name, ok, _ in RESULTS:
    print(f"{name:<40} {'PASS' if ok else 'FAIL'}")
print(f"\n{sum(ok for _, ok, _ in RESULTS)}/{len(RESULTS)} contracts pass")

contract                                 result
----------------------------------------------------
T1 ledger = 1 bit                        PASS
T1 ledger is the low counter bit         PASS
T1 junk bit not added                    PASS
T1 poke = run                            PASS
T2 ledger = 1 bit                        PASS
T2 ledger is the carry                   PASS
T2 no ledger without a future            PASS
T2 peek/poke = run                       PASS
T3 factoring threshold                   PASS
T4 capacity rule                         PASS
T5a hosting law                          PASS
T5b ell_min = ell (review)               FAIL
T5c ell_min = ell iff module             PASS
T5d ledger = count per group             PASS
T5e c>ell target-only class              PASS

14/15 contracts pass


**Established here, nothing beyond it:**

- **Regression.** The build-1 contracts rerun on the branching-capable module with the same numbers: 10/10.
- **Test 5 splits the capacity law in two.**
  - **Hosting holds** (T5a, 622/622). At $c<\ell$ source configurations are excluded. At $c=\ell$ every source configuration and every source move has an image, and nothing is invented. At $c>\ell$ the receiver has states the source never writes, and one move from a written state reaches them (T5e, 304/304).
  - **Continuation does not follow from capacity** (T5b FAIL on 300 of 622 cases). Occupancy at $c=\ell$ is continuation-faithful exactly when $M$ is a module: every vertex outside $M$ is adjacent to all of $M$ or to none of it (T5c, 622/622). When $M$ is not a module, two source configurations with the same receiver reading differ on one move, at every capacity. So $\ell_{\min}=\ell$ for modules, and no capacity makes occupancy faithful otherwise. This is the review's FAIL branch: "a different ledger shape than occupancy alone."
- **Ledger shape** (T5d, 622/622). Group the vertices of $M$ by what they see outside $M$. The minimal ledger is the token count in each group.
  - A module is one group, so its ledger is plain occupancy.
  - An edge whose endpoints see different outsides has two groups, so its ledger is which endpoint holds the token.

**Receiver law, as tested.** Capacity $c=\ell$ is necessary for a faithful contraction. It is sufficient only when $M$ is a module. In general the receiver must carry one count per outside-neighbourhood group of $M$, not one count for the merged site.

**Scope.** Graphs of up to 10 vertices, exclusion tokens, $K\le 4$, contracted sets of 2–3 vertices. The partition refinement is Moore's algorithm, which becomes bisimulation for branching sources; minimality is Myhill–Nerode. The contribution is the runtime contract, not the algorithm.